# 04 - Baseline Models Evaluation (B0, A0, B2, B3 across 3 seeds)

**Đồ án môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (COVID-19 Radiography Database).  
**Sprint 2 - Tuần 4:** Thiết lập toàn diện các mốc mô hình cơ sở trên tập **Validation** (tuyệt đối KHÔNG đụng tập Test):
- **B0:** Mô hình luôn đoán lớp đa số (`Normal`).
- **A0:** Mô hình phi bệnh lý (đo lường lối tắt phân loại dựa trên độ sáng và kênh màu).
- **B2:** ResNet18 Head-only (đóng băng backbone, 3 seeds: 42, 43, 44, 5 epochs).
- **B3:** ResNet18 Full Fine-tuning (3 seeds: 42, 43, 44, 5 epochs, trọng số ImageNet official).

> **Quy tắc nghiêm ngặt:** Chỉ đánh giá trên tập Validation; `allow_test` luôn là `False`; không xử lý mất cân bằng ở giai đoạn này.

In [ ]:
# 1. Khởi tạo môi trường, module src/ và định vị dữ liệu
import os
import sys
import time
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU: {torch.cuda.get_device_name(0)}")
    print(f"Số lượng GPU: {torch.cuda.device_count()}")

os.makedirs("src", exist_ok=True)
os.makedirs("splits", exist_ok=True)
os.makedirs("results/b0_majority", exist_ok=True)
os.makedirs("results/a0_shortcut", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("outputs/checkpoints", exist_ok=True)

with open("src/__init__.py", "w", encoding="utf-8") as f: f.write("")
with open("src/dataset.py", "w", encoding="utf-8") as f: f.write('''"""Module dataset.py

Trách nhiệm:
- Quản lý nạp dữ liệu từ file split_v1.csv (phụ trách: Nguyễn Đức).
- Định nghĩa lớp PyTorch CXRDataset cho bài toán phân loại 4 nhóm ảnh X-quang ngực.
- Thiết kế Data Augmentation cho tập Train và Transform chuẩn hóa cho Validation (phụ trách: Vũ Thương).
- Cài đặt cơ chế bảo vệ nghiêm ngặt: Khóa tập Test (allow_test=False).
- Tính toán Class Weights hỗ trợ xử lý mất cân bằng lớp.
"""

import os
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# 4 lớp bài toán phân loại X-quang ngực
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
LABEL_TO_IDX: Dict[str, int] = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_LABEL: Dict[int, str] = {idx: name for idx, name in enumerate(CLASS_NAMES)}

# Chuẩn hóa ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_transforms(
    img_size: int = 224,
    is_train: bool = True,
) -> transforms.Compose:
    """Tạo chuỗi biến đổi tiền xử lý và tăng cường dữ liệu ảnh X-quang.

    Quy tắc y khoa nghiêm ngặt (Vũ Thương phụ trách):
    - TUYỆT ĐỐI KHÔNG sử dụng RandomHorizontalFlip (lật ngang).
      Lý do: Tim người nằm ở lồng ngực bên trái; lật ngang làm đảo ngược vị trí tim và
      bóng các cơ quan nội tạng (Situs Inversus nhân tạo), gây sai lệch đặc trưng y học.
    - Train augmentation an toàn: Xoay nhẹ (+/- 7 độ), co giãn nhẹ (scale 0.95-1.05),
      chỉnh sáng và tương phản nhẹ (+/- 10%).
    - Validation: Chỉ resize và normalize.

    Args:
        img_size (int): Kích thước cạnh ảnh đầu ra (mặc định 224).
        is_train (bool): True nếu áp dụng augmentation cho tập train, False cho val/test.

    Returns:
        transforms.Compose: Pipeline tiền xử lý của torchvision.
    """
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomRotation(degrees=(-7, 7)),
            transforms.RandomAffine(
                degrees=0,
                translate=(0.02, 0.02),
                scale=(0.95, 1.05),
            ),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


class CXRDataset(Dataset):
    """Lớp Dataset nạp ảnh X-quang ngực từ split_v1.csv.

    Hỗ trợ cơ chế khóa tập test: mặc định cấm truy cập split='test' (allow_test=False).
    """

    def __init__(
        self,
        split_df_or_path: Union[pd.DataFrame, str, Path],
        split: str = "train",
        data_root: Optional[Union[str, Path]] = None,
        transform: Optional[Callable] = None,
        allow_test: bool = False,
    ) -> None:
        split_lower = split.lower().strip()
        if split_lower not in {"train", "val", "test"}:
            raise ValueError(f"Split không hợp lệ: '{split}'. Chỉ chấp nhận 'train', 'val', hoặc 'test'.")

        if split_lower == "test" and not allow_test:
            raise PermissionError(
                "TẬP TEST ĐÃ BỊ KHÓA NGHIÊM NGẶT!\\n"
                "Theo quy tắc đạo đức nghiên cứu và chống rò rỉ dữ liệu của đồ án:\\n"
                "- Không được phép truy cập, đánh giá hoặc tinh chỉnh siêu tham số trên tập test ở Tuần 3-9.\\n"
                "- Mọi quyết định lựa chọn mô hình chỉ được thực hiện trên tập Validation.\\n"
                "- Tập Test chỉ được mở DUY NHẤT một lần tại Tuần 10 trên mô hình tối ưu đã chốt."
            )

        if isinstance(split_df_or_path, (str, Path)):
            df_full = pd.read_csv(split_df_or_path)
        else:
            df_full = split_df_or_path.copy()

        self.df = df_full[df_full["split"] == split_lower].reset_index(drop=True)
        self.split = split_lower
        self.data_root = Path(data_root) if data_root is not None else None

        if transform is not None:
            self.transform = transform
        else:
            self.transform = get_transforms(img_size=224, is_train=(split_lower == "train"))

    def __len__(self) -> int:
        return len(self.df)

    def _resolve_image_path(self, row: pd.Series) -> Path:
        if self.data_root is not None:
            candidate = self.data_root / row["rel_image_path"]
            if candidate.exists():
                return candidate

        candidate = Path(row["rel_image_path"])
        if candidate.exists():
            return candidate

        if "image_path" in row and pd.notna(row["image_path"]):
            candidate = Path(row["image_path"])
            if candidate.exists():
                return candidate

        raise FileNotFoundError(
            f"Không thể tìm thấy file ảnh: {row['image_id']} (rel: {row['rel_image_path']})"
        )

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row = self.df.iloc[idx]
        img_path = self._resolve_image_path(row)

        with Image.open(img_path) as img:
            img = img.convert("RGB")
            tensor_img = self.transform(img)

        label_name = row["label"]
        label_idx = LABEL_TO_IDX[label_name]
        return tensor_img, label_idx


def build_dataloaders(
    split_csv_path: Union[str, Path],
    data_root: Optional[Union[str, Path]] = None,
    batch_size: int = 32,
    num_workers: int = 2,
    img_size: int = 224,
    allow_test: bool = False,
) -> Dict[str, DataLoader]:
    train_ds = CXRDataset(
        split_df_or_path=split_csv_path,
        split="train",
        data_root=data_root,
        transform=get_transforms(img_size=img_size, is_train=True),
        allow_test=False,
    )
    val_ds = CXRDataset(
        split_df_or_path=split_csv_path,
        split="val",
        data_root=data_root,
        transform=get_transforms(img_size=img_size, is_train=False),
        allow_test=False,
    )
    dataloaders = {
        "train": DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=(num_workers > 0),
        ),
        "val": DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=(num_workers > 0),
        ),
    }
    return dataloaders
''')
with open("src/models.py", "w", encoding="utf-8") as f: f.write('''"""Module models.py

Trách nhiệm:
- Định nghĩa và khởi tạo kiến trúc mạng học sâu cho bài toán phân loại X-quang 4 lớp.
- Hỗ trợ mô hình ResNet-18 (tối thiểu cho Smoke Test Tuần 3), mở rộng DenseNet-121 và EfficientNet-B0.
- Ghi nhận phân công: Thái Tú phụ trách chính, Vũ Thương kèm cặp duyệt chéo.
"""

import torch
import torch.nn as nn
from torchvision import models


def get_model(
    model_name: str = "resnet18",
    num_classes: int = 4,
    pretrained: bool = False,
) -> nn.Module:
    name_clean = model_name.lower().replace("-", "").replace("_", "")
    if "resnet18" in name_clean:
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif "densenet121" in name_clean:
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
    elif "efficientnetb0" in name_clean:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
    else:
        raise ValueError(f"Kiến trúc '{model_name}' chưa được hỗ trợ!")
    return model


def freeze_backbone(model: nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = False
    if hasattr(model, "fc"):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif hasattr(model, "classifier"):
        for param in model.classifier.parameters():
            param.requires_grad = True


def unfreeze_all(model: nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = True
''')
with open("src/metrics.py", "w", encoding="utf-8") as f: f.write('''"""Module metrics.py

Trách nhiệm:
- Tính toán toàn diện các chỉ số đánh giá cho bài toán phân loại X-quang 4 lớp (phụ trách: Hải Lý).
"""

from typing import Dict, List, Optional, Union
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

DEFAULT_CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]


def compute_metrics(
    y_true: Union[List[int], np.ndarray],
    y_pred: Union[List[int], np.ndarray],
    class_names: Optional[List[str]] = None,
) -> Dict[str, Union[float, Dict[str, float], List[List[int]]]]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if class_names is None:
        class_names = DEFAULT_CLASS_NAMES
    num_classes = len(class_names)

    acc = float(accuracy_score(y_true, y_pred))
    balanced_acc = float(balanced_accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    weighted_f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    macro_prec = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    macro_rec = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    per_class_prec = precision_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)
    per_class_rec = recall_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)
    per_class_f1 = f1_score(y_true, y_pred, average=None, labels=list(range(num_classes)), zero_division=0)

    per_class_dict = {}
    for idx, name in enumerate(class_names):
        support_count = int(np.sum(y_true == idx))
        per_class_dict[name] = {
            "precision": float(per_class_prec[idx]),
            "recall": float(per_class_rec[idx]),
            "f1": float(per_class_f1[idx]),
            "support": support_count,
        }

    return {
        "accuracy": round(acc, 4),
        "balanced_accuracy": round(balanced_acc, 4),
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
        "macro_precision": round(macro_prec, 4),
        "macro_recall": round(macro_rec, 4),
        "per_class": per_class_dict,
        "confusion_matrix": cm.tolist(),
    }
''')
with open("src/train.py", "w", encoding="utf-8") as f: f.write('''"""Module train.py

Trách nhiệm:
- Điều khiển vòng lặp huấn luyện (train loop) và kiểm định (val loop) cho mô hình (phụ trách: Minh Thuận).
"""

import json
from pathlib import Path
import time
from typing import Any, Dict, Optional, Tuple
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from src.metrics import compute_metrics


def save_checkpoint(state: Dict[str, Any], filepath: Path) -> None:
    filepath.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, str(filepath))


def load_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    scaler: Optional[Any] = None,
    device: str = "cpu",
) -> Tuple[int, float]:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file checkpoint: {checkpoint_path}")
    checkpoint = torch.load(str(checkpoint_path), map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and "scheduler_state_dict" in checkpoint and checkpoint["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    saved_epoch = checkpoint.get("epoch", 0)
    best_val_f1 = checkpoint.get("best_val_f1", checkpoint.get("val_macro_f1", 0.0))
    return saved_epoch + 1, best_val_f1


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: Optional[Any],
    device: torch.device,
    use_amp: bool = True,
) -> Tuple[float, float]:
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    for images, targets in dataloader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        if use_amp and device.type == "cuda" and scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        _, preds = torch.max(outputs, 1)
        correct_preds += torch.sum(preds == targets.data).item()
        total_samples += batch_size
    return running_loss / total_samples, correct_preds / total_samples


def validate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    class_names: Optional[list] = None,
) -> Tuple[float, Dict[str, Any]]:
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_targets = []
    all_preds = []
    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, targets)
            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size
            _, preds = torch.max(outputs, 1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    val_loss = running_loss / total_samples
    val_metrics = compute_metrics(all_targets, all_preds, class_names=class_names)
    return val_loss, val_metrics


def train_pipeline(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[Any] = None,
    num_epochs: int = 5,
    device_name: str = "cuda" if torch.cuda.is_available() else "cpu",
    output_dir: Path = Path("outputs/checkpoints"),
    run_id: str = "run",
    use_amp: bool = True,
    class_names: Optional[list] = None,
) -> Dict[str, Any]:
    device = torch.device(device_name)
    model.to(device)
    checkpoint_dir = output_dir / run_id
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    last_pt_path = checkpoint_dir / "last.pt"
    best_pt_path = checkpoint_dir / "best.pt"
    history_path = checkpoint_dir / "train_history.json"
    scaler = torch.cuda.amp.GradScaler() if (use_amp and device.type == "cuda") else None
    best_val_f1 = 0.0
    history = []

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(
            model=model, dataloader=train_loader, criterion=criterion, optimizer=optimizer, scaler=scaler, device=device, use_amp=use_amp
        )
        val_loss, val_metrics = validate(
            model=model, dataloader=val_loader, criterion=criterion, device=device, class_names=class_names
        )
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
        elapsed = time.time() - t0
        val_f1 = val_metrics["macro_f1"]
        val_acc = val_metrics["accuracy"]
        print(f"  Epoch {epoch:02d} [{elapsed:.1f}s]: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
        ckpt_state = {
            "epoch": epoch,
            "run_id": run_id,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_f1": val_f1,
            "best_val_f1": max(best_val_f1, val_f1),
        }
        save_checkpoint(ckpt_state, last_pt_path)
        is_best = val_f1 > best_val_f1
        if is_best:
            best_val_f1 = val_f1
            save_checkpoint(ckpt_state, best_pt_path)
        history.append({
            "epoch": epoch,
            "train_loss": round(train_loss, 4),
            "train_acc": round(train_acc, 4),
            "val_loss": round(val_loss, 4),
            "val_acc": round(val_acc, 4),
            "val_macro_f1": round(val_f1, 4),
            "is_best": is_best,
            "elapsed_seconds": round(elapsed, 2),
        })
    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)
    return {
        "run_id": run_id,
        "best_val_f1": best_val_f1,
        "last_checkpoint": str(last_pt_path),
        "best_checkpoint": str(best_pt_path),
        "history": history,
    }
''')
with open("src/utils.py", "w", encoding="utf-8") as f: f.write('''"""Module utils.py"""
import os
import random
import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
''')

input_root = Path("/kaggle/input")
if not Path("splits/split_v1.csv").exists():
    for p in input_root.rglob("split_v1.csv"):
        shutil.copy(p, "splits/split_v1.csv")
        print(f"Đã sao chép split_v1.csv từ: {p}")
        break
if not Path("splits/manifest.csv").exists():
    for p in input_root.rglob("manifest.csv"):
        shutil.copy(p, "splits/manifest.csv")
        print(f"Đã sao chép manifest.csv từ: {p}")
        break
assert Path("splits/split_v1.csv").exists(), "Không tìm thấy split_v1.csv!"
print("Đã thiết lập đầy đủ mã nguồn và dữ liệu split_v1.csv!")


In [ ]:
# 2. MÔ HÌNH CƠ SỞ B0: Luôn dự đoán lớp đa số của tập Train
from src.metrics import compute_metrics, DEFAULT_CLASS_NAMES
from src.utils import set_seed

set_seed(42)
split_df = pd.read_csv("splits/split_v1.csv")
train_df = split_df[split_df["split"] == "train"]
val_df = split_df[split_df["split"] == "val"]

majority_class = train_df["label"].value_counts().index[0]
majority_idx = DEFAULT_CLASS_NAMES.index(majority_class)
print(f"Lớp đa số trong tập Train: {majority_class} (chỉ số {majority_idx})")

y_true_val = [DEFAULT_CLASS_NAMES.index(lbl) for lbl in val_df["label"]]
y_pred_b0 = [majority_idx] * len(val_df)

metrics_b0 = compute_metrics(y_true_val, y_pred_b0, class_names=DEFAULT_CLASS_NAMES)

print("\n=== KẾT QUẢ MÔ HÌNH CƠ SỞ B0 TRÊN TẬP VALIDATION ===")
print(f"Accuracy          : {metrics_b0['accuracy']:.4f}")
print(f"Balanced Accuracy : {metrics_b0['balanced_accuracy']:.4f}")
print(f"Macro-F1          : {metrics_b0['macro_f1']:.4f}")
for c, vals in metrics_b0['per_class'].items():
    print(f"  Lớp {c:<16}: Recall = {vals['recall']:.4f}, Precision = {vals['precision']:.4f}, F1 = {vals['f1']:.4f}")

with open("results/b0_majority/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_b0, f, indent=2, ensure_ascii=False)


In [ ]:
# 3. MÔ HÌNH PHI BỆNH LÝ A0: Học lối tắt từ độ sáng trung bình và kênh màu
if Path("splits/manifest.csv").exists():
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler

    manifest_df = pd.read_csv("splits/manifest.csv")
    merged = split_df.merge(manifest_df[["image_id", "color_mode", "mean_brightness"]], on="image_id")
    merged["is_rgb"] = (merged["color_mode"] == "RGB").astype(float)
    merged["brightness"] = merged["mean_brightness"].astype(float)

    train_m = merged[merged["split"] == "train"]
    val_m = merged[merged["split"] == "val"]

    feats = ["brightness", "is_rgb"]
    X_tr = train_m[feats].values
    y_tr = np.array([DEFAULT_CLASS_NAMES.index(l) for l in train_m["label"]])
    X_va = val_m[feats].values
    y_va = np.array([DEFAULT_CLASS_NAMES.index(l) for l in val_m["label"]])

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)

    clf_lr = LogisticRegression(random_state=42, max_iter=1000)
    clf_lr.fit(X_tr_s, y_tr)
    y_pred_a0_lr = clf_lr.predict(X_va_s)
    metrics_a0_lr = compute_metrics(y_va, y_pred_a0_lr, class_names=DEFAULT_CLASS_NAMES)

    clf_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    clf_rf.fit(X_tr, y_tr)
    y_pred_a0_rf = clf_rf.predict(X_va)
    metrics_a0_rf = compute_metrics(y_va, y_pred_a0_rf, class_names=DEFAULT_CLASS_NAMES)

    print("\n=== KẾT QUẢ MÔ HÌNH PHI BỆNH LÝ A0 TRÊN TẬP VALIDATION ===")
    print(f"A0 (Logistic Regression) Macro-F1: {metrics_a0_lr['macro_f1']:.4f} | Balanced Acc: {metrics_a0_lr['balanced_accuracy']:.4f}")
    print(f"A0 (Random Forest depth=5) Macro-F1: {metrics_a0_rf['macro_f1']:.4f} | Balanced Acc: {metrics_a0_rf['balanced_accuracy']:.4f}")

    with open("results/a0_shortcut/metrics.json", "w", encoding="utf-8") as f:
        json.dump({"logistic_regression": metrics_a0_lr, "random_forest": metrics_a0_rf}, f, indent=2, ensure_ascii=False)


In [ ]:
# 4. CHẠY THỰC NGHIỆM B2 (HEAD-ONLY) VÀ B3 (FULL FINE-TUNE) TRÊN 3 SEEDS
import torch.nn as nn
from src.models import get_model, freeze_backbone, unfreeze_all
from src.dataset import build_dataloaders
from src.train import train_pipeline, validate, load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị tính toán: {device}")

dataset_root = None
for p in Path("/kaggle/input").rglob("COVID"):
    if p.is_dir() and (p / "images").exists():
        dataset_root = p.parent
        break
if dataset_root is None:
    dataset_root = Path("/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset")
print(f"Dataset root: {dataset_root}")

seeds = [42, 43, 44]
configs = [
    {"name": "B2_ResNet18_HeadOnly", "strategy": "head_only", "lr": 1e-3},
    {"name": "B3_ResNet18_FullFineTune", "strategy": "full_finetune", "lr": 1e-4},
]

records = []
all_run_metrics = {}

for cfg in configs:
    strategy = cfg["strategy"]
    lr = cfg["lr"]
    cfg_name = cfg["name"]
    
    for seed in seeds:
        run_id = f"{strategy}_resnet18_seed{seed}"
        print(f"\n{'='*70}")
        print(f"BẮT ĐẦU: {cfg_name} | Seed: {seed} | Run ID: {run_id}")
        print(f"{'='*70}")
        
        set_seed(seed)
        dataloaders = build_dataloaders(
            split_csv_path="splits/split_v1.csv",
            data_root=dataset_root,
            batch_size=32,
            num_workers=2,
            img_size=224,
            allow_test=False,
        )
        
        model = get_model(model_name="resnet18", num_classes=4, pretrained=True)
        if strategy == "head_only":
            freeze_backbone(model)
            params = [p for p in model.parameters() if p.requires_grad]
        else:
            unfreeze_all(model)
            params = model.parameters()
            
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)
        
        t_start = time.time()
        res = train_pipeline(
            model=model,
            train_loader=dataloaders["train"],
            val_loader=dataloaders["val"],
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            num_epochs=5,
            device_name=str(device),
            output_dir=Path("outputs/checkpoints"),
            run_id=run_id,
            use_amp=True,
            class_names=DEFAULT_CLASS_NAMES,
        )
        total_sec = time.time() - t_start
        gpu_hours = total_sec / 3600.0
        
        # Đánh giá chi tiết mô hình tốt nhất với model_best chuyển sang GPU (.to(device))
        best_model = get_model(model_name="resnet18", num_classes=4, pretrained=False).to(device)
        load_checkpoint(Path(res["best_checkpoint"]), best_model, device=str(device))
        val_loss, val_metrics = validate(best_model, dataloaders["val"], criterion, device, class_names=DEFAULT_CLASS_NAMES)
        
        rec = {
            "run_id": run_id,
            "model": "resnet18",
            "strategy": strategy,
            "seed": seed,
            "epochs": 5,
            "batch_size": 32,
            "learning_rate": lr,
            "total_seconds": round(total_sec, 2),
            "gpu_hours": round(gpu_hours, 4),
            "val_loss": round(val_loss, 4),
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_acc": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "covid_f1": val_metrics["per_class"]["COVID"]["f1"],
            "lung_opacity_f1": val_metrics["per_class"]["Lung_Opacity"]["f1"],
            "normal_f1": val_metrics["per_class"]["Normal"]["f1"],
            "viral_pneumonia_f1": val_metrics["per_class"]["Viral Pneumonia"]["f1"],
        }
        records.append(rec)
        all_run_metrics[run_id] = {
            "summary": rec,
            "best_checkpoint": res["best_checkpoint"],
            "metrics": val_metrics,
            "history": res["history"],
        }
        print(f"\nKẾT THÚC {run_id}: Val Macro-F1 = {val_metrics['macro_f1']:.4f} | Time: {total_sec:.1f}s ({gpu_hours:.4f} GPU-h)")

# 5. TỔNG HỢP VÀ XUẤT KẾT QUẢ
df_results = pd.DataFrame(records)
df_results.to_csv("results/baseline_summary.csv", index=False)
print("Đã lưu bảng kết quả tại: results/baseline_summary.csv")

with open("results/baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_run_metrics, f, indent=2, ensure_ascii=False)
print("Đã lưu chi tiết metrics tại: results/baseline_metrics.json")

print("\n" + "="*80)
print("BẢNG TỔNG HỢP KẾT QUẢ BASELINE TRÊN TẬP VALIDATION")
print("="*80)
print(df_results[["run_id", "strategy", "seed", "val_macro_f1", "val_accuracy", "val_balanced_acc", "gpu_hours"]].to_string())
